# Model Input/Outputt


- main：
    - promt template
    - llms
    - chatmodel
    - output parser

[LangChain开发入门教程](https://github.com/QunBB/DeepLearning/tree/main/llms)

连接模型以及分词器

In [39]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langchain_community.embeddings import HuggingFaceEmbeddings

load_dotenv("apikey.env")
BASE_URL = 'https://api.deepseek.com'
API_KEY = os.getenv('DEEPSEEK-API-KEY')
deepseek_chat_model = 'deepseek-chat'
if  not API_KEY:
    raise ValueError("WARNING: NOT FOUND OPENAI_API_KEY，PLEASE CHECK .env SETING。")
else:
    print("SECESSFULLY!")

model = ChatDeepSeek(api_key=API_KEY, base_url=BASE_URL, model=deepseek_chat_model)
embeddings = HuggingFaceEmbeddings(model_name="moka-ai/m3e-base")

SECESSFULLY!


In [40]:
model, embeddings

(ChatDeepSeek(client=<openai.resources.chat.completions.completions.Completions object at 0x000001E21BD38460>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001E21BD380D0>, root_client=<openai.OpenAI object at 0x000001E21BD39810>, root_async_client=<openai.AsyncOpenAI object at 0x000001E21BD39CC0>, model_name='deepseek-chat', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.deepseek.com', api_key=SecretStr('**********'), api_base='https://api.deepseek.com/v1'),
 HuggingFaceEmbeddings(client=SentenceTransformer(
   (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
   (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
 ), mo

##  $\large{PromptTemplate}$

In [7]:
from langchain.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "你是一名笑话高手，请你说一个关于{topic}的笑话，并且给笑话打分总分为{total_score}"
)

chain = prompt | model

print(prompt.invoke({"topic" : "哈基米", "total_score":"10" }))

for chunck in chain.stream({"topic" : "哈基米", "total_score":"10" }):
    print(chunck.content, end="", flush=True)

text='你是一名笑话高手，请你说一个关于哈基米的笑话，并且给笑话打分总分为10'
当然！作为一名笑话高手，我为您精心创作了一个关于“哈基米”的笑话，并附上专业评分：

---

### 笑话：《哈基米的职业规划》

小猫咪哈基米决定去找工作。面试官问它：“你的特长是什么？”  
哈基米挺起胸膛，骄傲地回答：“我特别擅长‘咪咪咪咪咪咪’，能连续唱10小时不跑调！”  
面试官震惊：“这算什么特长？”  
哈基米歪头一笑：“您不知道吗？我的歌在短视频平台有**几亿播放量**，人类听了都会跟着扭腰——这叫**顶级流量运营**！”  
面试官沉默三秒，当场递上合同：“明天来上班，职位是**首席洗脑旋律官**！”

---

### 笑话评分：
- **创意分**：9/10（结合网络热梗与职场幽默，跨界联动）  
- **节奏分**：8/10（对话推进自然，包袱在结尾爆发）  
- **接地气分**：10/10（精准捕捉“哈基米”魔性旋律的全民记忆）  
- **综合得分**：**9分**（能让人一边笑一边想起被BGM支配的快乐）  

---

希望这只职场精英哈基米能逗您一笑！😸

## $\large{ChatPromptTemplate}$

In [18]:
from langchain_core.prompts import ChatPromptTemplate

chatPrompt = ChatPromptTemplate(
    [
        ("system", "你是一个{personality}类型的聊天机器人， 名字叫{name}"), 
        ("user", "你好，{user_input}"), 
    ]
)

message = chatPrompt.format_messages(personality="暴躁老哥", 
                                     name="张三", 
                                     user_input="今天过得怎么样？")

chatPrompt_chain = chatPrompt | model 

print(message)

input_dict = {
    "personality": "暴躁老哥", 
    "name": "张三", 
    "user_input": "今天过得怎么样？"
}

for chunck in chatPrompt_chain.stream(input_dict):
    print(chunck.content, end="", flush=True)

[SystemMessage(content='你是一个暴躁老哥类型的聊天机器人， 名字叫张三', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，今天过得怎么样？', additional_kwargs={}, response_metadata={})]
（不耐烦地）别整这些虚头巴脑的，有事说事！老子忙着呢！

## 缓存$cache$

In [19]:
from langchain.globals import set_llm_cache
from langchain.cache import InMemoryCache


In [41]:
model_1 = model.bind(temperature=0.0)

In [52]:
%%time

set_llm_cache(InMemoryCache())

# The first time, it is not yet in cache, so it should take longer
for chunck in model_1.stream("说一个笑话"):
    print(chunck.content, end="", flush=True)


一个程序员去商店买牛奶，妻子发短信说：“买一升牛奶，如果看到鸡蛋，买六个。”  

结果程序员带着六升牛奶回家了。  
妻子惊讶地问：“你为什么买了六升牛奶？”  

程序员回答：“因为我看到鸡蛋了。”CPU times: total: 46.9 ms
Wall time: 3.67 s


In [53]:
%%time

# The second time it is, so it goes faster
for chunck in model_1.stream("说一个笑话"):
    print(chunck.content, end="", flush=True)

一个程序员去商店买东西。  
店员问他：“您需要袋子吗？”  
程序员回答：“嗯…等等，我查一下文档。”CPU times: total: 31.2 ms
Wall time: 2.65 s


## 记录消耗的tokens

In [57]:
from langchain_community.callbacks import get_openai_callback
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
        SystemMessage(content="你是一个数学专家"),
        HumanMessage(content="勾股定理的定义？"),
    ]

with get_openai_callback() as cb:
    for chunck in model.stream(messages):
        print(chunck.content, end="", flush=True)
    print(cb)

勾股定理是平面几何中的基本定理之一，其定义如下：

**在一个直角三角形中，斜边的平方等于两条直角边的平方和。**

用数学公式表达为：  
若直角三角形的两条直角边长度分别为 \(a\) 和 \(b\)，斜边长度为 \(c\)，则满足：
\[
a^2 + b^2 = c^2
\]

**补充说明：**
1. 勾股定理仅适用于**直角三角形**。
2. 定理的逆命题也成立：若一个三角形的三边满足 \(a^2 + b^2 = c^2\)，则该三角形为直角三角形，且 \(c\) 为斜边。
3. 勾股定理有超过400种不同的证明方法，包括几何证明、代数证明等，最早可追溯到古巴比伦时期。

如果需要进一步了解其证明或应用，可以继续提问！Tokens Used: 195
	Prompt Tokens: 13
		Prompt Tokens Cached: 0
	Completion Tokens: 182
		Reasoning Tokens: 0
Successful Requests: 1
Total Cost (USD): $0.0


# 如何解析 JSON 输出
虽然一些大模型供应商支持 内置方式返回结构化输出，但并非所有都支持。

JsonOutputParser 是一个内置选项，用于提示和解析 JSON 输出。

虽然它的功能与 PydanticOutputParser 类似，但它还支持流式返回部分 JSON 对象。
## 使用 `JsonOutputParser`

In [68]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from langchain_deepseek import ChatDeepSeek

llm_deepseek = ChatDeepSeek(model=deepseek_chat_model, 
                            api_key=API_KEY, base_url=BASE_URL, 
                            temperature=1.3)

# Define your desired data structure.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

# Set up a parser + inject instructions into the prompt template.
parser = JsonOutputParser(pydantic_object=Joke)

prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],

    # 请注意，我们将 format_instructions 从解析器直接传递到提示中。
    # 也可以自定义增强或者替换默认指令
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | llm_deepseek | parser


# And a query intented to prompt a language model to populate the data structure.
joke_query = "给我一个“热”笑话。"
for chunck in chain.stream({"query": joke_query}):
    print(chunck, flush=True)

{}
{'setup': ''}
{'setup': '为什么'}
{'setup': '为什么太阳'}
{'setup': '为什么太阳从来不'}
{'setup': '为什么太阳从来不单独'}
{'setup': '为什么太阳从来不单独行动'}
{'setup': '为什么太阳从来不单独行动？'}
{'setup': '为什么太阳从来不单独行动？', 'punchline': ''}
{'setup': '为什么太阳从来不单独行动？', 'punchline': '因为它'}
{'setup': '为什么太阳从来不单独行动？', 'punchline': '因为它总是'}
{'setup': '为什么太阳从来不单独行动？', 'punchline': '因为它总是带着'}
{'setup': '为什么太阳从来不单独行动？', 'punchline': '因为它总是带着热度'}
{'setup': '为什么太阳从来不单独行动？', 'punchline': '因为它总是带着热度（'}
{'setup': '为什么太阳从来不单独行动？', 'punchline': '因为它总是带着热度（热'}
{'setup': '为什么太阳从来不单独行动？', 'punchline': '因为它总是带着热度（热得'}
{'setup': '为什么太阳从来不单独行动？', 'punchline': '因为它总是带着热度（热得不行'}
{'setup': '为什么太阳从来不单独行动？', 'punchline': '因为它总是带着热度（热得不行）'}
{'setup': '为什么太阳从来不单独行动？', 'punchline': '因为它总是带着热度（热得不行）！'}


In [67]:
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"setup": {"description": "question to set up a joke", "title": "Setup", "type": "string"}, "punchline": {"description": "answer to resolve the joke", "title": "Punchline", "type": "string"}}, "required": ["setup", "punchline"]}\n```'

## 无需 Pydantic


这将提示模型返回 JSON，但不会提供关于模式应是什么的具体信息。(不会必定输出为结构化信息)

In [69]:
joke_query = "Tell me a joke."

parser = JsonOutputParser()

prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | llm_deepseek | parser

chain.invoke({"query": joke_query})

{'joke': "Why don't scientists trust atoms? Because they make up everything."}

# 自定义解析器
- 实现自定义解析器有两种方法：

    - 使用 LCEL 中的 `RunnableLambda` or `RunnableGenerator` （推荐）
    - 通过从一个基础类继承来进行输出解析（复杂）

(这两种方法之间的区别主要是表面的，主要体现在触发的回调（例如，on_chain_start 与 on_parser_start）以及在像 LangSmith 这样的追踪平台中，如何可视化可运行的 lambda 与解析器。)
    

## 可运行的 Lambda 和生成器
推荐的解析方式是使用 可运行的 lambda 和 可运行的生成器！

In [72]:
from typing import Iterable
from langchain_core.messages import AIMessage, AIMessageChunk

def parse(ai_message: AIMessage) -> str:
    """Parse the AI message."""
    return ai_message.content.swapcase()


chain = llm_deepseek | parse
print((chain.invoke("hello")))

for chunk in chain.stream("tell me about yourself in one sentence"):
    print(chunk, end="|", flush=True)

hELLO! 👋 hOW CAN i HELP YOU TODAY?
i AM dEEPsEEK, AN ai ASSISTANT CREATED BY dEEPsEEK cOMPANY TO PROVIDE HELPFUL AND HARMLESS RESPONSES TO YOUR QUESTIONS.|

这种方式的流式处理无效，因为解析器**在解析输出之前会聚合输入**

In [74]:
from langchain_core.runnables import RunnableGenerator

def streaming_parse(chunks: Iterable[AIMessageChunk]) -> Iterable[str]:
    for chunk in chunks:
        yield chunk.content.swapcase()

streaming_parse = RunnableGenerator(streaming_parse)

chain = model | streaming_parse

for chunk in chain.stream("tell me about yourself in one sentence"):
    print(chunk, end="|", flush=True)

|i| AM| dEEP|sE|EK|,| AN| ai| ASSISTANT| CREATED| BY| dEEP|sE|EK| cOMPANY| TO| PROVIDE| HELPFUL| AND| HARMLESS| RESPONSES| TO| YOUR| QUERIES|.||

## 从解析基类继承
[pass, please read docs](https://www.langchain.com.cn/docs/how_to/output_parser_custom/)